In [1]:
import torch
from pathlib import Path
from torch_geometric.nn import to_hetero

from halide_gnn_cost_model.data import PipelineDataset
from halide_gnn_cost_model.model import PipeGCN, PipeGAT, PipelineModel

In [2]:
PIPELINES_DIR = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/pipelines-test")
GCN_MODEL_PATH = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/models/gcn/gcn_l1_epoch_100.pt")
GAT_MODEL_PATH = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/models/gat/gat_l1_epoch_100.pt")

In [3]:
USE_GPU = True

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_GPU and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

Using device: mps


In [4]:
# Load dataset
dataset = PipelineDataset(PIPELINES_DIR)
data = dataset[0]  # Get the first pipeline graph
len(dataset)

1lines [00:00, 2409.13lines/s]
1lines [00:00, 40329.85lines/s]


2157

# Eval Metrics

In [5]:
def average_runtime_error(model, dataset):
    model.eval()
    total_error = 0
    with torch.no_grad():
        for graph in dataset:
            pred_log = model(graph)
            pred = torch.exp(pred_log)
            true = graph.y
            error = torch.abs(pred - true) / true
            total_error += error.mean().item()
    return total_error / len(dataset)

# GCN Eval

In [6]:
DIM_EMBEDDING = 64
gcn = PipeGCN(hidden_channels=DIM_EMBEDDING, out_channels=DIM_EMBEDDING, num_layers=4)
gcn = to_hetero(gcn, data.metadata(), aggr="sum")
model = PipelineModel(gcn, DIM_EMBEDDING, 5, len(dataset.ast_vocab), len(dataset.sched_vocab))
model.load_state_dict(torch.load(GCN_MODEL_PATH, map_location=device))

/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'drop

<All keys matched successfully>

## Avergate Runtime Error

In [7]:
print(f"Average Runtime Error on Test Set: {average_runtime_error(model, dataset) * 100:.2f}%")

Average Runtime Error on Test Set: 51.70%


## Plot Prediction Error Distribution

## Performance for Different Input Scales

# GAT Eval

In [8]:
DIM_EMBEDDING = 64
gat = PipeGAT(hidden_channels=DIM_EMBEDDING, out_channels=DIM_EMBEDDING, num_layers=4)
gat = to_hetero(gat, data.metadata(), aggr="sum")
model = PipelineModel(gat, DIM_EMBEDDING, 5, len(dataset.ast_vocab), len(dataset.sched_vocab))
model.load_state_dict(torch.load(GAT_MODEL_PATH, map_location=device))

/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'drop

<All keys matched successfully>

In [9]:
print(f"Average Runtime Error on Test Set: {average_runtime_error(model, dataset) * 100:.2f}%")

Average Runtime Error on Test Set: 53.01%
